# HR SQL workbook · Kestrel Media Co.

Kestrel Media Co. is a **fictional** company (all data simulated by `generate_hr_data.py`). It has eight HR tables shaped like a real HRIS: an employee master, effective-dated job and pay history, reviews, engagement pulses, a mentoring program, and terminations. The data was pulled on **2026-06-30**.

**How to use this workbook**
1. Read the question. Write your query in the `myQuery` string. Run the cell.
2. `check()` tells you if your result matches. Column **order** matters; column names and row order don't.
3. Stuck? `hint("2.2")` gives one hint at a time (`hint("2.2", 1)`). Still stuck after 15 minutes? `solution("2.2")`.
4. **Try first, hint second, solution last.** Struggling for a few minutes is where the learning happens. Reading a solution feels productive but sticks much less.

The SQL here is DuckDB, which is close to Snowflake, BigQuery, and Postgres. Where something is DuckDB-specific (like `QUALIFY`), the hints say so.

See `schema.md` for every table and column.

In [1]:
import os
import subprocess
import sys

# generate the data the first time (takes a few seconds)
if not os.path.exists("data/employees.csv"):
    subprocess.run([sys.executable, "generate_hr_data.py"], check=True)

from sqlkit import connect, run, check, question, hint, solution

con = connect()
run("SHOW TABLES")

,name
0,compensation
1,employees
2,engagement_survey
3,job_history
4,mentoring_enrollments
5,performance_reviews
6,salary_bands
7,terminations


## Get to know the data first

Before writing any analysis, look at every table: row counts, a few rows, and the grain (what one row means). **Ten minutes here saves hours of wrong joins later.**

In [2]:
for t in ["employees", "job_history", "compensation", "salary_bands", "performance_reviews",
          "engagement_survey", "mentoring_enrollments", "terminations"]:
    n = run(f"SELECT count(*) AS n FROM {t}").n[0]
    print(f"{t:<24}{n:>8,} rows")

employees                  6,200 rows
job_history                7,951 rows
compensation              17,359 rows
salary_bands                  42 rows
performance_reviews       12,492 rows
engagement_survey         30,024 rows
mentoring_enrollments      1,154 rows
terminations               2,895 rows


In [3]:
run("SELECT * FROM job_history WHERE employee_id = 100050 ORDER BY effective_date")

,employee_id,effective_date,department,job_level,manager_id,change_type
0,100050,2014-07-08,Talent,1,100031,Hire


One employee, several rows: hired, then promoted or transferred. That's an **effective-dated** table. The grain is one row per job change, not one row per person. Joining it straight to `employees` multiplies rows, which is the most common bug in HR SQL.

---
## Level 1 · Warm-up
SELECT, WHERE, GROUP BY, ORDER BY.

### 1.1 · Hires per year

Count hires by calendar year.  
   Columns: hire_year, hires

In [4]:
myQuery = """
-- your query here

"""
check("1.1", myQuery)
# hint("1.1")      # solution("1.1")

[1.1] not attempted yet


### 1.2 · Exits by year and type

Count terminations by year of termination and termination_type (Voluntary / Involuntary).  
   Columns: term_year, termination_type, exits

In [5]:
myQuery = """
-- your query here

"""
check("1.2", myQuery)
# hint("1.2")      # solution("1.2")

[1.2] not attempted yet


### 1.3 · Experience by hire source

Average prior_experience_years for each hire_source, rounded to 1 decimal.  
   Columns: hire_source, avg_prior_experience

In [6]:
myQuery = """
-- your query here

"""
check("1.3", myQuery)
# hint("1.3")      # solution("1.3")

[1.3] not attempted yet


---
## Level 2 · Joins
LEFT JOIN, anti-joins, conditional counts. Most HR questions need at least one join, and most wrong answers come from the wrong join type.

### 2.1 · Active headcount by location

How many employees are active as of 2026-06-30, by location?  
   Active = no row in terminations (every termination in the data is on or before 2026-06-30).  
   Columns: location, active_headcount

In [7]:
myQuery = """
-- your query here

"""
check("2.1", myQuery)
# hint("2.1")      # solution("2.1")

[2.1] not attempted yet


**Think:** what would happen to your count if you used an INNER JOIN instead?

### 2.2 · Early voluntary exits by hire source

For people hired 2020-01-01 through 2025-06-30, what share left VOLUNTARILY within 365 days of hire?  
   Early voluntary exit = termination_type 'Voluntary' and termination_date < hire_date + 365 days.  
   Columns: hire_source, hires, early_vol_exits, rate_pct (percent, 1 decimal)

In [8]:
myQuery = """
-- your query here

"""
check("2.2", myQuery)
# hint("2.2")      # solution("2.2")

[2.2] not attempted yet


### 2.3 · Survey response rate

Response rate for the 2025-12-15 engagement pulse.  
   Eligible = hired on or before 2025-11-15 AND not terminated on or before 2025-12-15.  
   Responded = eligible and has a row in engagement_survey with survey_date = 2025-12-15.  
   Columns: eligible, responded, response_rate_pct (1 decimal)

In [9]:
myQuery = """
-- your query here

"""
check("2.3", myQuery)
# hint("2.3")      # solution("2.3")

[2.3] not attempted yet


**Think:** move the `survey_date` condition from ON to WHERE and rerun. Why does `eligible` change? This is one of the most common SQL bugs in HR reporting.

---
## Level 3 · Dates and headcount
Point-in-time headcount and turnover rates, the bread and butter of people analytics. Interviewers love these because the date logic is easy to get subtly wrong.

### 3.1 · Month-end headcount, 2025

Headcount on the last day of each month in 2025.  
   On payroll on day X = hire_date <= X AND (no termination OR termination_date > X).  
   Columns: month_end, headcount

In [10]:
myQuery = """
-- your query here

"""
check("3.1", myQuery)
# hint("3.1")      # solution("3.1")

[3.1] not attempted yet


### 3.2 · 2025 voluntary turnover rate

Annual voluntary turnover for 2025 = voluntary exits in 2025 / average of the 12 month-end headcounts.  
   Columns: vol_exits, avg_headcount (1 decimal), turnover_pct (1 decimal)

In [11]:
myQuery = """
-- your query here

"""
check("3.2", myQuery)
# hint("3.2")      # solution("3.2")

[3.2] not attempted yet


**Think:** why divide by the *average* headcount instead of the January or December headcount? Turnover definitions differ across companies, so in an interview, always say which one you used.

### 3.3 · Tenure bands

Active employees (as of 2026-06-30) by tenure band.  
   Tenure in years = date_diff('day', hire_date, DATE '2026-06-30') / 365.25  
   Bands: '<1 yr' (under 1), '1-2 yrs' (1 to under 2), '2-5 yrs' (2 to under 5), '5+ yrs'.  
   Columns: tenure_band, headcount

In [12]:
myQuery = """
-- your query here

"""
check("3.3", myQuery)
# hint("3.3")      # solution("3.3")

[3.3] not attempted yet


---
## Level 4 · Window functions
ROW_NUMBER, LAG, rolling averages. Effective-dated HR tables (one row per change) are exactly what window functions are for.

### 4.1 · Current department and level

job_history has one row per job change. An employee's CURRENT job is their latest row (by effective_date).  
   For active employees, count headcount by current department and current job_level.  
   Columns: department, job_level, headcount

In [13]:
myQuery = """
-- your query here

"""
check("4.1", myQuery)
# hint("4.1")      # solution("4.1")

[4.1] not attempted yet


### 4.2 · Promotions per year (using LAG)

A promotion = a job_history row whose job_level is higher than that employee's previous row.  
   Don't use change_type; the point is to practice LAG. Count promotions by year of effective_date.  
   Columns: promo_year, promotions

In [14]:
myQuery = """
-- your query here

"""
check("4.2", myQuery)
# hint("4.2")      # solution("4.2")

[4.2] not attempted yet


**Think:** compare your answer to `SELECT year(effective_date), count(*) FROM job_history WHERE change_type = 'Promotion' GROUP BY 1`. Same? In real HRIS data they often aren't (data entry, demotions, level re-mapping), which is why analysts derive events from the data instead of trusting labels.

### 4.3 · Time to first promotion

For employees who were ever promoted, months from hire to FIRST promotion.  
   Report the median by department at hire (the department on their 'Hire' row).  
   Months = date_diff('month', hire_date, first_promotion_date).  
   Columns: department, promoted_employees, median_months

In [15]:
myQuery = """
-- your query here

"""
check("4.3", myQuery)
# hint("4.3")      # solution("4.3")

[4.3] not attempted yet


### 4.4 · Rolling 3-month voluntary exits, 2025

Voluntary exits per month in 2025, plus a rolling average of the current and previous two months  
   (for January use January alone; for February, Jan-Feb). Months with zero exits must still appear.  
   Columns: month_start, vol_exits, rolling_3mo_avg (2 decimals)

In [16]:
myQuery = """
-- your query here

"""
check("4.4", myQuery)
# hint("4.4")      # solution("4.4")

[4.4] not attempted yet


### 4.5 · Latest rating for active employees

For active employees with at least one review, take their most recent review (highest review_year).  
   Count them by that review's rating (NULL rating = review not completed; keep those as their own group).  
   Columns: latest_rating, employees

In [17]:
myQuery = """
-- your query here

"""
check("4.5", myQuery)
# hint("4.5")      # solution("4.5")

[4.5] not attempted yet


**Think:** the NULL group is employees whose latest review wasn't completed. If you dropped them, what would happen to an analysis of ratings and attrition? (Remember the missing-rating feature at Ingredion.)

---
## Level 5 · Cohorts and retention
Retention by hire cohort, retention curves, and small-group suppression.

### 5.1 · 12-month retention by hire cohort

For hire years 2020 through 2024, the share of hires still employed 365 days after hire.  
   Left within a year = termination_date < hire_date + 365 days (any termination type).  
   Columns: hire_year, hires, retained_12m, retention_pct (1 decimal)

In [18]:
myQuery = """
-- your query here

"""
check("5.1", myQuery)
# hint("5.1")      # solution("5.1")

[5.1] not attempted yet


**Think:** why stop at the 2024 cohort? What would go wrong if you included 2025 hires?

### 5.2 · Retention curve for the 2023 cohort

For people hired in 2023: the percent still employed at 3, 6, 12, 18, and 24 months.  
   Still employed at m months = no termination_date before hire_date + INTERVAL m MONTH.  
   Columns: months, pct_retained (1 decimal)

In [19]:
myQuery = """
-- your query here

"""
check("5.2", myQuery)
# hint("5.2")      # solution("5.2")

[5.2] not attempted yet


### 5.3 · First-year voluntary attrition by gender and location, with suppression

Hires 2022-01-01 through 2025-06-30. For each gender x location group:  
   hires, early voluntary exits (Voluntary, within 365 days), and rate_pct (1 decimal).  
   Small-group rule: if a group has fewer than 30 hires, rate_pct must be NULL (suppressed).  
   Columns: gender, location, hires, early_vol_exits, rate_pct

In [20]:
myQuery = """
-- your query here

"""
check("5.3", myQuery)
# hint("5.3")      # solution("5.3")

[5.3] not attempted yet


**Think:** Woman/Remote and Man/Remote rates differ by several points. Is that a real gender gap? How would you test it, and what would you say to a leader before you had tested it?

---
## Level 6 · Capstone
Build a research-ready analysis table from six tables. You'll use it in `02_mentoring_effect.ipynb` to estimate whether mentoring works.

### 6.1 · Capstone: build the mentoring analysis table

One row per employee hired 2023-01-01 through 2025-06-30 (everyone in this window has a full year of data).  
   Columns, in this order:  
     employee_id  
     mentored                 1 if in mentoring_enrollments, else 0  
     hire_source, location, gender, prior_experience_years  
     department_at_hire       from their 'Hire' row in job_history  
     level_at_hire            from their 'Hire' row in job_history  
     comp_ratio_at_hire       hire salary ('Hire' row in compensation) / band_mid for that department and level,  
                              rounded to 3 decimals  
     left_within_12m          1 if any termination_date < hire_date + 365 days, else 0  
     first_pulse_engagement   average of the four survey items on their FIRST survey response after hire  
                              (NULL if they never responded), rounded to 2 decimals  
   This is the table you'll use to estimate the mentoring program's effect.

In [21]:
myQuery = """
-- your query here

"""
check("6.1", myQuery)
# hint("6.1")      # solution("6.1")

[6.1] not attempted yet


**Check:** your table should have exactly one row per hire in the window. If it has more, a join is fanning out (which table has more than one row per employee?).

---
## Done? What to take into an interview

- **Grain first.** Before any join, say out loud what one row means in each table.
- **Define before you compute.** Turnover, retention, and "active" all have several reasonable definitions. Stating yours is half the answer.
- **Point-in-time logic.** Headcount on a date is `hired on or before AND (not terminated OR terminated after)`. Practice saying it.
- **Window functions** are how you handle effective-dated tables: latest row (`ROW_NUMBER`), previous row (`LAG`), running numbers (frames).
- **Suppress small groups.** Rates on tiny groups are noisy and can identify people.

Next: open `02_mentoring_effect.ipynb` and use your capstone table to estimate whether mentoring reduces first-year attrition.